In [19]:
import os
import pandas as pd

In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
## PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE

## svm
from sklearn.svm import SVC
import pandas as pd


In [22]:
ML_CLASSIFIER = ['MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", "XGBoost"] # "ELM"

def classify(X_train, y_train, X_test, y_test, classifier, search_type='grid'):
    """
    Classify the data using a Random Forest Classifier
    :param X_train: Training data
    :param y_train: Training labels
    :param X_test: Testing data
    :param y_test: Testing labels
    :param search_type: Type of search to perform
    :return: Accuracy of the classifier
    """
    # Create a Random Forest Classifier
    #clf = RandomForestClassifier()

    if classifier == "MLP":
        # clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000) ### uncomment if the performance is not good with below hyperparameter
        clf = MLPClassifier(hidden_layer_sizes=(100,100, 50), max_iter=1000)
    elif classifier == "GaussianNB": ### {'priors': None, 'var_smoothing': 1e-05}
        clf = GaussianNB(priors=None, var_smoothing= 1e-05) ### Often, the default parameters work well for many datasets.
    elif classifier == "Adaboost":
        dtc = DecisionTreeClassifier(ccp_alpha=0.0, class_weight='balanced', criterion='entropy', max_depth=10, max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0, min_samples_leaf=2, min_samples_split=2, splitter='best')
        clf = AdaBoostClassifier(estimator=dtc, n_estimators=200, random_state=0, learning_rate=1) ###{'learning_rate': 1, 'n_estimators': 200}
        
    elif classifier == "KNN":
        clf = KNeighborsClassifier(algorithm='auto', leaf_size=10, metric='euclidean', n_jobs=-1,  n_neighbors=1, p=1, weights='uniform')  
    elif classifier == "RFClassifier": ### 
        clf = RandomForestClassifier(bootstrap = False, criterion = 'entropy', max_depth = None, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 2, n_estimators = 100, oob_score = False, random_state = 42)
    elif classifier == "SVM_linear": ##
        clf = SVC(kernel='linear')
    elif classifier == "SVM_sigmoid": ###
        clf = SVC(kernel='sigmoid')
    elif classifier == "SVM_RBF": 
        clf = SVC(C=10, cache_size = 200, class_weight = None, gamma = 'scale', kernel = 'rbf', max_iter = -1, probability = True, shrinking = True, tol = 0.001)
    elif classifier == "XGBoost": #'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.7
        clf = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3,subsample=0.7)
    else:
        print("Invalid classifier")
        return None
        # Fit the model
    clf.fit(X_train, y_train)

    # Predict the test data
    y_pred = clf.predict(X_test)

    # Calculate the accuracy
    accuracy = balanced_accuracy_score(y_test, y_pred)
    
    return accuracy


In [24]:
data = pd.read_csv('BT-large-2c-dataset_results_finetune_ALL_Models_v2.csv')
data.head(15)

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,resnet50,0.766667,0.825000,0.785000,0.758333,0.796667,0.726667,0.841667,0.518333,0.880000
1,resnet101,0.785000,0.836667,0.705000,0.775000,0.848333,0.718333,0.830000,0.541667,0.873333
2,densenet121,0.938333,0.970000,0.850000,0.975000,0.981667,0.941667,0.960000,0.953333,0.973333
3,densenet169,0.955000,0.970000,0.818333,0.975000,0.980000,0.938333,0.968333,0.955000,0.978333
4,vgg16,0.928333,0.973333,0.830000,0.970000,0.963333,0.933333,0.970000,0.913333,0.981667
5,vgg19,0.918333,0.973333,0.736667,0.968333,0.961667,0.938333,0.970000,0.833333,0.980000
6,alexnet,0.926667,0.975000,0.825000,0.975000,0.978333,0.958333,0.963333,0.618333,0.983333
7,resnext50_32x4d,0.748333,0.816667,0.590000,0.738333,0.771667,0.706667,0.813333,0.483333,0.838333
8,resnext101_32x8d,0.808333,0.858333,0.650000,0.826667,0.913333,0.761667,0.863333,0.651667,0.906667
9,shufflenet_v2_x1_0,0.871667,0.965000,0.720000,0.931667,0.976667,0.881667,0.965000,0.948333,0.973333


In [25]:
## taking row-wise average of all the data 
data['average'] = data.drop("Model", axis=1).mean(axis=1)


In [26]:
data.drop("Model", axis=1).mean(axis=0)

XGBoost         0.914133
MLP             0.956933
GaussianNB      0.805133
Adaboost        0.938533
KNN             0.954200
RFClassifier    0.916267
SVM_linear      0.949733
SVM_sigmoid     0.854933
SVM_RBF         0.965600
average         0.917274
dtype: float64

In [7]:
## add coumn-wise average of all the data
#data.loc['average'] = data.drop("Model", axis=1).mean(axis=0)

In [27]:
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
0,resnet50,0.766667,0.825000,0.785000,0.758333,0.796667,0.726667,0.841667,0.518333,0.880000,0.766481
1,resnet101,0.785000,0.836667,0.705000,0.775000,0.848333,0.718333,0.830000,0.541667,0.873333,0.768148
2,densenet121,0.938333,0.970000,0.850000,0.975000,0.981667,0.941667,0.960000,0.953333,0.973333,0.949259
3,densenet169,0.955000,0.970000,0.818333,0.975000,0.980000,0.938333,0.968333,0.955000,0.978333,0.948704
4,vgg16,0.928333,0.973333,0.830000,0.970000,0.963333,0.933333,0.970000,0.913333,0.981667,0.940370
5,vgg19,0.918333,0.973333,0.736667,0.968333,0.961667,0.938333,0.970000,0.833333,0.980000,0.920000
6,alexnet,0.926667,0.975000,0.825000,0.975000,0.978333,0.958333,0.963333,0.618333,0.983333,0.911481
7,resnext50_32x4d,0.748333,0.816667,0.590000,0.738333,0.771667,0.706667,0.813333,0.483333,0.838333,0.722963
8,resnext101_32x8d,0.808333,0.858333,0.650000,0.826667,0.913333,0.761667,0.863333,0.651667,0.906667,0.804444
9,shufflenet_v2_x1_0,0.871667,0.965000,0.720000,0.931667,0.976667,0.881667,0.965000,0.948333,0.973333,0.914815


In [28]:
## sort the data by average
data = data.sort_values(by='average', ascending=False)
data

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF,average
14,vit_large_patch16_224,0.985000,0.996667,0.868333,0.991667,0.985000,0.983333,0.996667,0.983333,0.996667,0.976296
24,vit_base_patch32_384,0.981667,0.991667,0.883333,0.993333,0.985000,0.988333,0.990000,0.965000,0.995000,0.974815
22,vit_small_patch32_384,0.976667,0.991667,0.906667,0.995000,0.990000,0.988333,0.975000,0.948333,0.996667,0.974259
13,vit_base_patch32_224,0.971667,0.991667,0.876667,0.990000,0.988333,0.980000,0.991667,0.963333,0.995000,0.972037
20,vit_base_patch16_384,0.975000,0.985000,0.895000,0.988333,0.986667,0.976667,0.985000,0.963333,0.985000,0.971111
15,vit_small_patch32_224,0.970000,0.985000,0.891667,0.990000,0.993333,0.968333,0.971667,0.936667,0.995000,0.966852
12,vit_base_patch16_224,0.973333,0.993333,0.865000,0.981667,0.986667,0.978333,0.991667,0.938333,0.990000,0.966481
23,vit_small_patch16_384,0.976667,0.991667,0.833333,0.990000,0.990000,0.990000,0.975000,0.945000,0.991667,0.964815
19,vit_small_patch16_224,0.970000,0.991667,0.858333,0.990000,0.985000,0.980000,0.975000,0.938333,0.993333,0.964630
17,vit_base_patch8_224,0.968333,0.993333,0.855000,0.985000,0.985000,0.973333,0.995000,0.881667,0.993333,0.958889


In [30]:
list(data.head(3)['Model'])

['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']

### Top 3 performance model

In [32]:
top_model_combinations = [
                          ['vit_large_patch16_224', 'vit_base_patch32_384'],
                          ['vit_large_patch16_224', 'vit_small_patch32_384'],
                          ['vit_base_patch32_384', 'vit_small_patch32_384'],
                          ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
                          ]

Ensemble vgg16 with Top 2 networks ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']

In [33]:



# top_model_combinations = [
#                           ['vit_base_patch32_384', 'vit_small_patch32_224'],
#                           ['vit_base_patch32_384', 'vgg16'],
#                           ['vit_small_patch32_224', 'vgg16'],
#                           ['vit_base_patch32_384', 'vit_small_patch32_224', 'vgg16']
#                           ]

In [37]:
columns = ['Model', "XGBoost", 'MLP', 'GaussianNB', "Adaboost", "KNN", "RFClassifier", "SVM_linear", "SVM_sigmoid", "SVM_RBF", ]

dataframe = pd.DataFrame(columns=columns)
# add 12 rows to the dataframe with zero values 
for model_list in top_model_combinations:
    model_name = ' + '.join(model_list)
    new_row = {'Model': model_name, "XGBoost":0, 'MLP': 0, 'GaussianNB': 0, "Adaboost": 0, "KNN": 0, "RFClassifier": 0, "SVM_linear": 0, "SVM_sigmoid": 0, "SVM_RBF": 0} 
    dataframe.loc[len(dataframe)] = new_row

model_versions = ['simple_version', "with_normalization_&_PCA", "with_SMOTE_only", "with_normalization_&_PCA_SMOTE"]

model_version_index = 3
main_path = 'extracted_features_BT-large-2c'
for ml_classifier in ML_CLASSIFIER:
    for model_list in top_model_combinations:
        print('Model List:', model_list)
        ensemble_X_train = []
        ensemble_X_test = []

        for model in model_list:
            sub_dir = os.path.join(main_path, model)
            # Load the data
            X_train = np.load(os.path.join(sub_dir, 'train_data_array_features.npy'))
            y_train = np.load(os.path.join(sub_dir, 'train_data_array_labels.npy'))
            X_test = np.load(os.path.join(sub_dir, 'test_data_array_features.npy'))
            y_test = np.load(os.path.join(sub_dir, 'test_data_array_labels.npy'))


            ## squeeze the dimensions 1 from the features 
            X_train = np.squeeze(X_train, axis=1)
            X_test = np.squeeze(X_test, axis=1)

            ## concatenate the features on axis 1
            ensemble_X_train.append(X_train)
            ensemble_X_test.append(X_test)
            

        X_train = np.concatenate(ensemble_X_train, axis=1)
        y_train = y_train
        X_test = np.concatenate(ensemble_X_test, axis=1)
        y_test = y_test

        # # apply the standard scaler and PCA
        scaler = StandardScaler()
        X_train_normalized = scaler.fit_transform(X_train)
        X_test_normalized = scaler.transform(X_test)

        # Step 2 & 3: Apply PCA
        n_components = int(0.45 * X_train.shape[1])
        pca = PCA(n_components=n_components)
        X_train = pca.fit_transform(X_train_normalized)
        X_test = pca.transform(X_test_normalized)

        ## no of example in each class
        print("no of example in each class before SMOTE:", np.bincount(y_train))


        # # ## over sample the data (data augmentation)
        smote = SMOTE(sampling_strategy='minority')
        X_train, y_train = smote.fit_resample(X_train, y_train)

        # ## no of example in each class
        print("no of example in each class after SMOTE:", np.bincount(y_train))


        # print("no of features in X_train:", X_train.shape)

        
        # Classify the data
        accuracy = classify(X_train, y_train, X_test, y_test, ml_classifier)
        print('Accuracy:', accuracy)
        model_name = ' + '.join(model_list)

        dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy

    print(dataframe)
    dataframe.to_csv(f'BT-large-2c-dataset_results_top_three_{model_versions[model_version_index]}.csv', index=False)


Model List: ['vit_large_patch16_224', 'vit_base_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9966666666666667
Model List: ['vit_large_patch16_224', 'vit_small_patch32_384']


C:\Users\user\AppData\Local\Temp\ipykernel_5768\3735964397.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9966666666666667' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9983333333333333
Model List: ['vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9933333333333333
Model List: ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.995
                                               Model  XGBoost       MLP  \
0       vit_large_patch16_224 + vit_base_patch32_384        0  0.996667   
1      vit_large_patch16_224 + vit_small_patch32_384        0  0.998333   
2       vit_base_patch32_384 + vit_small_patch32_384        0  0.993333   
3  vit_large_patch16_224 + vit_base_patch32_384 +...        0  0.995000   

   GaussianNB  Adaboost  KNN  RFClassifier  SVM_linear  SVM_sigmoid  SVM_RBF  
0

C:\Users\user\AppData\Local\Temp\ipykernel_5768\3735964397.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6666666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.6983333333333333
Model List: ['vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.725
Model List: ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.6783333333333333
                                               Model  XGBoost       MLP  \
0       vit_large_patch16_224 + vit_base_patch32_384        0  0.996667   
1      vit_large_patch16_224 + vit_small_patch32_384        0  0.998333   
2       vit_base_patch32_384 + vit_small_patch32_384        0  0.993333   
3  vit_large_patch16_224 + vit_base_patch32_384 +...        0  0.995000   

   GaussianNB  Adaboost  KNN  RFClassifier  SVM_linear  SVM_sigmoid  SVM_RBF  
0

...

no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9650000000000001
Model List: ['vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9783333333333333
Model List: ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.98
                                               Model  XGBoost       MLP  \
0       vit_large_patch16_224 + vit_base_patch32_384        0  0.996667   
1      vit_large_patch16_224 + vit_small_patch32_384        0  0.998333   
2       vit_base_patch32_384 + vit_small_patch32_384        0  0.993333   
3  vit_large_patch16_224 + vit_base_patch32_384 +...        0  0.995000   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0    

C:\Users\user\AppData\Local\Temp\ipykernel_5768\3735964397.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9966666666666667' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9966666666666667
Model List: ['vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.995
Model List: ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9966666666666667
                                               Model  XGBoost       MLP  \
0       vit_large_patch16_224 + vit_base_patch32_384        0  0.996667   
1      vit_large_patch16_224 + vit_small_patch32_384        0  0.998333   
2       vit_base_patch32_384 + vit_small_patch32_384        0  0.993333   
3  vit_large_patch16_224 + vit_base_patch32_384 +...        0  0.995000   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0   

C:\Users\user\AppData\Local\Temp\ipykernel_5768\3735964397.py:74: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9883333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataframe.loc[dataframe['Model'] == model_name, ml_classifier] = accuracy


no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.99
Model List: ['vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.985
Model List: ['vit_large_patch16_224', 'vit_base_patch32_384', 'vit_small_patch32_384']
no of example in each class before SMOTE: [4227 4268]
no of example in each class after SMOTE: [4268 4268]
Accuracy: 0.9933333333333334
                                               Model   XGBoost       MLP  \
0       vit_large_patch16_224 + vit_base_patch32_384  0.988333  0.996667   
1      vit_large_patch16_224 + vit_small_patch32_384  0.990000  0.998333   
2       vit_base_patch32_384 + vit_small_patch32_384  0.985000  0.993333   
3  vit_large_patch16_224 + vit_base_patch32_384 +...  0.993333  0.995000   

   GaussianNB  Adaboost       KNN  RFClassifier  SVM_linear  SVM_sigmoid  \
0    0.666667

In [19]:
n_components

768

In [14]:
dataframe

,Model,XGBoost,MLP,GaussianNB,Adaboost,KNN,RFClassifier,SVM_linear,SVM_sigmoid,SVM_RBF
0,vit_base_patch32_384 + vit_small_patch32_224,0.736086,0.762759,0.334131,0.731610,0.771137,0.738784,0.799189,0.683773,0.754797
1,vit_base_patch32_384 + vit_small_patch16_224,0.733586,0.758041,0.333893,0.754932,0.796542,0.748919,0.779797,0.658766,0.754797
2,vit_small_patch32_224 + vit_small_patch16_224,0.740533,0.756554,0.410204,0.764571,0.812421,0.777488,0.719953,0.619690,0.765811
3,vit_base_patch32_384 + vit_small_patch32_224 +...,0.721848,0.767432,0.325231,0.747432,0.802421,0.782297,0.771554,0.665940,0.762297
